# Pre-fine-tuning model validation — Qwen3-0.6B

Run the selected Hugging Face model locally against the curated **validation** split, then score its JSON predictions with the project evaluator. The notebook downloads only `curated/validation.jsonl` and caches the model through Hugging Face. Start with a small sample; set `SAMPLE_SIZE = None` to evaluate the full split.

> This is a baseline before fine-tuning. Keep the test split for final evaluation. Semantic comparisons in the evaluator may require `TYPESAFE_API_KEY` in the project `.env`.

In [ ]:
from importlib import import_module
from pathlib import Path
import json
import os
import random
import sys

import torch
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download, snapshot_download
from huggingface_hub.errors import EntryNotFoundError
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src/steps/04_evaluation/evaluator.py').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Open this notebook from inside the project.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env', override=False)

evaluation = import_module('src.steps.04_evaluation')
EvaluationItem = evaluation.EvaluationItem
Evaluator = evaluation.Evaluator
ModelCallbackOutput = evaluation.ModelCallbackOutput
render_json_prompt = evaluation.render_json_prompt
report_results = evaluation.report_results

MODEL_ID = 'Qwen/Qwen3-0.6B'
DATASET_FILE = PROJECT_ROOT / '.data/curated/validation.jsonl'
SAMPLE_SIZE, SEED, MAX_NEW_TOKENS = 5, 42, 4096
WORKERS = 1  # Keep one generation at a time on a local device.

## 1. Download and load only the validation split

In [ ]:
def dataset_repo_id():
    repo_id = os.getenv('HF_DATASET_REPO', 'md-zain/resume-parser-dataset').strip()
    if '/' not in repo_id:
        owner = os.getenv('HF_USERNAME', 'md-zain').strip()
        repo_id = f'{owner}/{repo_id}'
    if repo_id.count('/') != 1:
        raise ValueError('Dataset repo must be owner/name (or set HF_USERNAME and HF_DATASET_REPO).')
    return repo_id


def download_validation_split():
    if DATASET_FILE.is_file():
        return DATASET_FILE

    repo_id = dataset_repo_id()
    revision = os.getenv('HF_DATASET_REVISION', 'main').strip() or 'main'
    token = os.getenv('HF_TOKEN') or None  # Public dataset: token is optional.
    DATASET_FILE.parent.mkdir(parents=True, exist_ok=True)
    print(f'Downloading only curated/validation.jsonl from {repo_id}...')
    try:
        hf_hub_download(
            repo_id=repo_id, repo_type='dataset', revision=revision,
            filename='curated/validation.jsonl', local_dir=str(PROJECT_ROOT / '.data'), token=token,
        )
    except EntryNotFoundError:
        # Also support the previous dataset layout with split files at repo root.
        hf_hub_download(
            repo_id=repo_id, repo_type='dataset', revision=revision,
            filename='validation.jsonl', local_dir=str(DATASET_FILE.parent), token=token,
        )
    if not DATASET_FILE.is_file():
        raise FileNotFoundError(f'Validation split was not created at {DATASET_FILE}')
    return DATASET_FILE


def load_validation_items(limit=SAMPLE_SIZE, seed=SEED):
    path = download_validation_split()
    with path.open(encoding='utf-8') as file:
        records = [json.loads(line) for line in file if line.strip()]
    if limit is not None:
        if limit < 1:
            raise ValueError('limit must be positive or None.')
        records = random.Random(seed).sample(records, min(limit, len(records)))
    return [
        EvaluationItem(
            resume_id=record['resume_id'],
            resume_text=record['resume_text'],
            target_json=record['target_json'],
            category=record.get('category'),
        )
        for record in records
    ]


items = load_validation_items()
print(f'Loaded {len(items)} validation resumes from {DATASET_FILE}')

## 2. Download and load Qwen3-0.6B locally

The model is downloaded once into the normal Hugging Face cache. Change `MODEL_ID` above later to try another Hub model.

In [ ]:
def download_model(model_id=MODEL_ID):
    return snapshot_download(repo_id=model_id, token=os.getenv('HF_TOKEN') or None)


def choose_device():
    if torch.cuda.is_available():
        return 'cuda'
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'


def load_model(model_path):
    device = choose_device()
    dtype = torch.float16 if device in {'cuda', 'mps'} else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=dtype)
    model.to(device).eval()
    return tokenizer, model, device


model_path = download_model()
tokenizer, model, DEVICE = load_model(model_path)
print(f'Loaded {MODEL_ID} on {DEVICE}')

## 3. Generate JSON and evaluate it

Generation uses the evaluator's existing JSON prompt. The evaluator validates JSON/schema and computes the project's programmatic and semantic scores.

In [ ]:
def generate_response(item):
    prompt = render_json_prompt(item.resume_text)
    messages = [{
        'role': 'user',
        'content': '/no_think\n' + prompt,
    }]
    try:
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        # Compatibility with Transformers versions whose template lacks enable_thinking.
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    inputs = tokenizer(formatted_prompt, return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs['input_ids'].shape[1]:]
    raw_json = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    if not raw_json:
        raise RuntimeError('Qwen3 returned an empty response.')
    return ModelCallbackOutput(
        raw_output=raw_json,
        prompt=formatted_prompt,
        model_name=MODEL_ID,
        metadata={'device': DEVICE, 'max_new_tokens': MAX_NEW_TOKENS},
    )

### 3a. Smoke-test one resume (no evaluator yet)

Run this cell first. It times one model response and prints the raw generated JSON, so you can confirm generation works before starting evaluation.

In [ ]:
from time import perf_counter

smoke_item = items[0]
started = perf_counter()
smoke_output = generate_response(smoke_item)
inference_seconds = perf_counter() - started

print(f'Resume: {smoke_item.resume_id}')
print(f'Inference time: {inference_seconds:.1f} seconds')
print('Generated JSON:')
print(smoke_output.raw_output)

### 3b. Score the smoke-test response

After the raw response looks reasonable, run the existing evaluator on that same single resume. The callback reuses the already-generated response for this resume, avoiding a second slow model call. For a broader run afterward, use `Evaluator(items)` with `generate_response` as the callback.

In [ ]:
def use_smoke_response(item):
    if item.resume_id == smoke_item.resume_id:
        return smoke_output
    return generate_response(item)


evaluator = Evaluator([smoke_item])
report = evaluator.evaluate(
    callback=use_smoke_response,
    workers=WORKERS,
)
print(json.dumps(report.metrics, indent=2))

## 4. View the evaluation report

The report displays the evaluator's charts and metrics. Saving is optional; run `summary.save()` in the next cell only if you want the report files written to `results/`.

In [ ]:
summary = report_results({f'huggingface:{MODEL_ID}': report})

In [ ]:
# Optional: save charts and metrics under results/.
# summary.save()